In [1]:
import numpy as np
import scipy as sp

import qiskit

import sys
sys.path.insert(0, '../')

import oasis_dla

In [2]:
def linear_indep(mats: list[np.ndarray]) -> bool:
    vecs = [m.flatten() for m in mats]
    M = np.column_stack(vecs)
    rank = np.linalg.matrix_rank(M)
    return rank == M.shape[1]

def commutator(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    return A @ B - B @ A

def construct_dla(generators):
    dla = [] # init empty dla to start

    # go through initial generators
    for gen in generators:
        # print(type(gen))
        pot_dla = [*dla, gen]
        print(pot_dla)
        if linear_indep(pot_dla):
            dla = pot_dla

    # now we're left w all the linear indep generators 
    # go thru all combinations of generators and compute brackets, find
    l = 1
    r = 0

    while l < len(dla):
        for m in range(r):
            bracket = commutator(dla[l], dla[m])
            pot_dla = [*dla, bracket]
            if linear_indep(pot_dla):
                dla = pot_dla
        r += 1
        if r == l:
            l += 1
            r = 0

    return dla
         




In [3]:
circuit = qiskit.QuantumCircuit(1)
circuit.h(0)
circuit.ry(0, 0)

gen = oasis_dla.extract_generators(circuit)

display(gen)

X = qiskit.circuit.library.XGate().to_matrix()
Y = qiskit.circuit.library.YGate().to_matrix()
Z = qiskit.circuit.library.ZGate().to_matrix()

dla = construct_dla(gen)

print(f"Computed DLA: \n{dla}")
print(f"dim(g): {len(dla)}")

[array([[-1.+0.j,  1.+0.j],
        [ 1.+0.j,  1.+0.j]]),
 array([[0.+0.j, 0.-1.j],
        [0.+1.j, 0.+0.j]])]

[array([[-1.+0.j,  1.+0.j],
       [ 1.+0.j,  1.+0.j]])]
[array([[-1.+0.j,  1.+0.j],
       [ 1.+0.j,  1.+0.j]]), array([[0.+0.j, 0.-1.j],
       [0.+1.j, 0.+0.j]])]
Computed DLA: 
[array([[-1.+0.j,  1.+0.j],
       [ 1.+0.j,  1.+0.j]]), array([[0.+0.j, 0.-1.j],
       [0.+1.j, 0.+0.j]])]
dim(g): 2


In [4]:
print(f"Var(cost) ~ 1/dim(g) = {1/len(dla)}")

Var(cost) ~ 1/dim(g) = 0.5


In [8]:
vqls = oasis_dla.VQLS()

A = np.array([
    [1, 1],
    [0, 1]
])

b = np.array([
    [1],
    [1]
])

X = np.array([
    [0, 1],
    [1, 0]
])

Y = np.array([
    [0, -1j],
    [1j, 0]
])

Z = np.array([
    [1, 0],
    [0, -1]
])

# pauli_matrices = [np.eye(2), X, Y, Z]
# coeffs = [0.5 * np.trace(P @ A) for P in pauli_matrices]

# sum = np.zeros((2, 2), np.complex128)
# for i, p in enumerate(pauli_matrices):
#     sum += (coeffs[i] * p)
# sum

solution = vqls.solve(A, b, shots=10_000)

print(solution.rescale_answer())
solution.plot_probabilities()
print(np.linalg.solve(A, b))


 message: The lower bound for the trust-region radius has been reached
 success: True
  status: 0
     fun: -1.0
       x: [ 1.571e+00]
     nit: 40
   maxcv: 0.0
    nfev: 18
[-7.95781715e-04+0.00290112j  9.99994423e-01-0.00145056j]
SamplerPubResult(data=DataBin(meas=BitArray(<shape=(), num_shots=10000, num_bits=1>)), metadata={'shots': 10000, 'circuit_metadata': {}})
[[0.]
 [1.]]


In [ ]:
prob_list = []
for i in range(0, 10):
    res = vqls.solve(A, b)
    prob_list.append(res.get_probabilites())

print(prob_list)